In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, make_scorer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV

In [2]:
target_train = pd.read_parquet("target_train.parquet")
network_train = pd.read_parquet("network_train.parquet")
weather_train = pd.read_parquet("weather_train.parquet")
weather_test = pd.read_parquet("weather_test.parquet")
network_test = pd.read_parquet("network_test.parquet")

In [10]:


### functions already used for EDA

def flatten_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ["_".join([str(i) for i in col]) for col in df.columns]
    return df


def interpolate_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_index()
    df = df.interpolate(method="time")
    df = df.ffill()
    return df


def aggregate_weather(df: pd.DataFrame) -> pd.DataFrame:
    ssrd_cols = [c for c in df.columns if c.endswith("ssrd")]
    tcc_cols  = [c for c in df.columns if c.endswith("tcc")]
    temp_cols = [c for c in df.columns if c.endswith("2t")]
    wind_cols = [c for c in df.columns if c.endswith("100ws")]

    flat = pd.DataFrame(index=df.index)
    flat["ssrd_mean"] = df[ssrd_cols].mean(axis=1)
    flat["ssrd_std"]  = df[ssrd_cols].std(axis=1)
    flat["tcc_mean"]  = df[tcc_cols].mean(axis=1)
    flat["tcc_std"]   = df[tcc_cols].std(axis=1)
    flat["temp_mean"] = df[temp_cols].mean(axis=1)
    flat["temp_std"] = df[temp_cols].std(axis=1)
    flat["wind_mean"] = df[wind_cols].mean(axis=1)
    flat["wind_std"]  = df[wind_cols].std(axis=1)

    return flat.bfill().ffill()


def prepare_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multi-index columns, interpolate missing values, and aggregate to per-hour stats."""
    df = flatten_weather_columns(df)
    df = interpolate_weather(df)
    df = aggregate_weather(df)
    df = df.reset_index().rename(columns={df.index.name or "index": "ds"})
    return df


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ssrd_lag1"]  = df["ssrd_mean"].shift(1)
    df["ssrd_lag24"] = df["ssrd_mean"].shift(24)
    return df


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ssrd_roll_3h"]     = df["ssrd_mean"].rolling(3).mean()
    df["ssrd_roll_24h"]    = df["ssrd_mean"].rolling(24).mean()
    df["tcc_roll_6h_std"]  = df["tcc_mean"].rolling(6).std()

    # Add new rolling features for tcc_mean, temp_mean, and wind_mean
    df["tcc_roll_3h_mean"] = df["tcc_mean"].rolling(3).mean()
    df["tcc_roll_24h_mean"] = df["tcc_mean"].rolling(24).mean()
    df["temp_roll_3h_mean"] = df["temp_mean"].rolling(3).mean()
    df["temp_roll_24h_mean"] = df["temp_mean"].rolling(24).mean()
    df["wind_roll_3h_mean"] = df["wind_mean"].rolling(3).mean()
    df["wind_roll_24h_mean"] = df["wind_mean"].rolling(24).mean()
    df["temp_roll_6h_std"]  = df["temp_mean"].rolling(6).std()
    df["wind_roll_6h_std"]  = df["wind_mean"].rolling(6).std()

    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(df["ds"])
    df["hour"]      = dt.dt.hour
    df["month"]     = dt.dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df



def engineer_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lags, rolling stats, time encodings, and solar features, then drop NaN rows."""
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_time_features(df)



    return df.dropna()
def prepare_solar_data(target_path: str, weather_path: str, network_path: str) -> pd.DataFrame:
    """Complete pipeline: load + preprocess weather + merge with targets and network features."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    # Interpolate and clip FR_solar_actual before merging
    target_train["FR_solar_actual"] = (
        target_train["FR_solar_actual"]
        .interpolate(method="time")
        .bfill()
        .ffill()
        .clip(lower=0)
    )

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    # Merge with targets and network features
    solar_df = target_train[['FR_solar_actual']].join(weather_features, how='inner').join(network_train, how='inner')

    return solar_df

def prepare_wind_data(target_path: str, weather_path: str) -> pd.DataFrame:
    """Complete pipeline for wind."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    wind_df = target_train[['FR_wind_actual']].join(weather_features)

    return wind_df

def prepare_load_data(target_path: str, weather_path: str, network_path: str) -> pd.DataFrame:
    """Complete pipeline for load (includes network features)."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    load_df = target_train[['FR_load_actual']].join(weather_features).join(network_train)

    return load_df
##for train test split
def split_data_chronologically(df: pd.DataFrame, test_size: float = 0.2):
    """
    Splits a DataFrame chronologically into training and validation sets,
    and separates features (X) from the target (y).

    Args:
        df (pd.DataFrame): The input DataFrame.
        test_size (float): The proportion of the dataset to include in the validation split.

    Returns:
        tuple: (X_train, y_train, X_valid, y_valid) as pandas DataFrames/Series.
    """
    # Calculate the split point
    split_point = int(len(df) * (1 - test_size))

    # Split the data chronologically
    train = df.iloc[:split_point]
    valid = df.iloc[split_point:]

    # Separate features (X) and target (y)
    X_train = train.drop('FR_solar_actual', axis=1)
    y_train = train['FR_solar_actual']
    X_valid = valid.drop('FR_solar_actual', axis=1)
    y_valid = valid['FR_solar_actual']

    print('Training set shape:', train.shape)
    print('Validation set shape:', valid.shape)

    return X_train, y_train, X_valid, y_valid

##for evaluation
def wmape(y_true, y_pred):
    """Weighted Mean Absolute Percentage Error"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

class TimeSeriesEvaluator:
    def compute_standard_metrics(self):
        return {
            'Target': self.target_name,
            'RMSE': np.sqrt(mean_squared_error(self.y_true, self.y_pred)),
            'MAE': mean_absolute_error(self.y_true, self.y_pred),
            'WMAPE': wmape(self.y_true, self.y_pred),
            'R²': r2_score(self.y_true, self.y_pred),
        }

In [40]:
weather_flat = prepare_weather(weather_train)
weather_flat.head()

,ds,ssrd_mean,ssrd_std,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std
0,2020-01-01 00:00:00+00:00,0.0,0.0,0.490648,0.425557,3.114028,3.679530,4.431250,2.351552
1,2020-01-01 01:00:00+00:00,0.0,0.0,0.499319,0.416623,2.807910,3.637566,4.399936,2.316953
2,2020-01-01 02:00:00+00:00,0.0,0.0,0.518816,0.414566,2.749704,3.592852,4.380681,2.291575
3,2020-01-01 03:00:00+00:00,0.0,0.0,0.534694,0.409608,2.656238,3.711994,4.475877,2.365991
4,2020-01-01 04:00:00+00:00,0.0,0.0,0.548571,0.407396,2.599818,3.779075,4.508040,2.380406


In [41]:
# Re-apply prepare_weather to get a clean DataFrame (with 'ds' as a column) as input for engineer_weather_features.
weather_flat = prepare_weather(weather_train)

# Now, apply the comprehensive feature engineering function
weather_flat = engineer_weather_features(weather_flat)

# Set 'ds' as the index for consistency with subsequent join operations, and remove the original 'ds' column.
weather_flat = weather_flat.set_index('ds')

print("Engineered weather_flat DataFrame head:")
display(weather_flat.head())
print("\nEngineered weather_flat DataFrame info:")
weather_flat.info()

Engineered weather_flat DataFrame head:


,ssrd_mean,ssrd_std,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,ssrd_lag1,ssrd_lag24,...,wind_roll_3h_mean,wind_roll_24h_mean,temp_roll_6h_std,wind_roll_6h_std,hour,month,hour_sin,hour_cos,month_sin,month_cos
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-02 00:00:00+00:00,0.0,0.0,0.591098,0.397544,4.024190,3.424526,3.861214,2.025915,0.0,0.0,...,3.762191,3.804964,0.148966,0.083488,0,1,0.000000,1.000000,0.5,0.866025
2020-01-02 01:00:00+00:00,0.0,0.0,0.626592,0.391434,3.880080,3.565624,4.081434,2.093134,0.0,0.0,...,3.893518,3.791693,0.102429,0.171253,1,1,0.258819,0.965926,0.5,0.866025
2020-01-02 02:00:00+00:00,0.0,0.0,0.647933,0.395149,3.885397,3.561708,4.182189,2.126657,0.0,0.0,...,4.041613,3.783423,0.076456,0.225824,2,1,0.500000,0.866025,0.5,0.866025
2020-01-02 03:00:00+00:00,0.0,0.0,0.652530,0.402927,3.899515,3.605973,4.303050,2.085833,0.0,0.0,...,4.188891,3.776222,0.059011,0.250485,3,1,0.707107,0.707107,0.5,0.866025
2020-01-02 04:00:00+00:00,0.0,0.0,0.651431,0.406992,3.928852,3.585760,4.358646,2.013896,0.0,0.0,...,4.281295,3.769997,0.054884,0.245929,4,1,0.866025,0.500000,0.5,0.866025



Engineered weather_flat DataFrame info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 43825 entries, 2020-01-02 00:00:00+00:00 to 2025-01-01 00:00:00+00:00
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ssrd_mean           43825 non-null  float32
 1   ssrd_std            43825 non-null  float32
 2   tcc_mean            43825 non-null  float32
 3   tcc_std             43825 non-null  float32
 4   temp_mean           43825 non-null  float32
 5   temp_std            43825 non-null  float32
 6   wind_mean           43825 non-null  float32
 7   wind_std            43825 non-null  float32
 8   ssrd_lag1           43825 non-null  float32
 9   ssrd_lag24          43825 non-null  float32
 10  ssrd_roll_3h        43825 non-null  float64
 11  ssrd_roll_24h       43825 non-null  float64
 12  tcc_roll_6h_std     43825 non-null  float64
 13  tcc_roll_3h_mean    43825 non-null  float64
 14  tcc_roll_24h_m

In [42]:
 # Drop original 'hour' and 'month' columns after creating cyclical features
weather_flat = weather_flat.drop(columns=['hour', 'month'])

In [43]:
weather_flat ['ssrd_log'] = np.log1p(weather_flat ['ssrd_mean'])
weather_flat ['ssrd_std_log'] = np.log1p(weather_flat ['ssrd_std'])

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [44]:
weather_flat = weather_flat.drop(columns=['ssrd_mean', 'ssrd_std'])

CREATE WIND DATAFRAME


In [45]:
X=weather_flat.copy()

In [46]:

target_train['FR_wind_actual']= (
    target_train['FR_wind_actual']
    .interpolate(method="time")
    .bfill()
    .ffill()
    .clip(lower=0)
)

y=target_train[['FR_wind_actual']]

In [47]:
wind_df = target_train[['FR_wind_actual']].join(X)
wind_df = wind_df.dropna()

In [48]:
wind_df

,FR_wind_actual,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,ssrd_lag1,ssrd_lag24,ssrd_roll_3h,...,wind_roll_3h_mean,wind_roll_24h_mean,temp_roll_6h_std,wind_roll_6h_std,hour_sin,hour_cos,month_sin,month_cos,ssrd_log,ssrd_std_log
start_date,,,,,,,,,,,,,,,,,,,,,
2020-01-02 00:00:00+00:00,1417.00,0.591098,0.397544,4.024190,3.424526,3.861214,2.025915,0.0,0.0,0.0,...,3.762191,3.804964,0.148966,0.083488,0.000000,1.000000,5.000000e-01,0.866025,0.0,0.0
2020-01-02 01:00:00+00:00,1465.00,0.626592,0.391434,3.880080,3.565624,4.081434,2.093134,0.0,0.0,0.0,...,3.893518,3.791693,0.102429,0.171253,0.258819,0.965926,5.000000e-01,0.866025,0.0,0.0
2020-01-02 02:00:00+00:00,1579.00,0.647933,0.395149,3.885397,3.561708,4.182189,2.126657,0.0,0.0,0.0,...,4.041613,3.783423,0.076456,0.225824,0.500000,0.866025,5.000000e-01,0.866025,0.0,0.0
2020-01-02 03:00:00+00:00,1646.00,0.652530,0.402927,3.899515,3.605973,4.303050,2.085833,0.0,0.0,0.0,...,4.188891,3.776222,0.059011,0.250485,0.707107,0.707107,5.000000e-01,0.866025,0.0,0.0
2020-01-02 04:00:00+00:00,1738.00,0.651431,0.406992,3.928852,3.585760,4.358646,2.013896,0.0,0.0,0.0,...,4.281295,3.769997,0.054884,0.245929,0.866025,0.500000,5.000000e-01,0.866025,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 20:00:00+00:00,9843.50,0.423017,0.451998,3.870347,3.879590,6.656932,3.539675,0.0,0.0,0.0,...,6.412070,4.578412,1.000897,0.509063,-0.866025,0.500000,-2.449294e-16,1.000000,0.0,0.0
2024-12-31 21:00:00+00:00,10179.25,0.419937,0.449693,3.623478,3.952355,6.903283,3.531578,0.0,0.0,0.0,...,6.656137,4.749639,0.681524,0.457719,-0.707107,0.707107,-2.449294e-16,1.000000,0.0,0.0
2024-12-31 22:00:00+00:00,10546.25,0.442860,0.455166,3.388112,3.972077,7.160623,3.558408,0.0,0.0,0.0,...,6.906946,4.926148,0.546603,0.454808,-0.500000,0.866025,-2.449294e-16,1.000000,0.0,0.0


Split data

In [49]:

X_train = wind_df.loc['2020':'2023'].drop('FR_wind_actual', axis=1)
y_train = wind_df.loc['2020':'2023']['FR_wind_actual']

X_valid = wind_df.loc['2024'].drop('FR_wind_actual', axis=1)
y_valid = wind_df.loc['2024']['FR_wind_actual']

 # TRAIN MODEL

## 1. LGBM

In [50]:


wind_model = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

wind_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse"
)




[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014420 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5417
[LightGBM] [Info] Number of data points in the train set: 34698, number of used features: 25
[LightGBM] [Info] Start training from score 4511.939830


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
              num_leaves=64, objective='regression', random_state=42,
              subsample=0.8)

EVALUATION

In [51]:

y_pred_train_lgbm = wind_model.predict(X_train)

mae_train_lgbm = mean_absolute_error(y_train, y_pred_train_lgbm)
rmse_train_lgbm = np.sqrt(mean_squared_error(y_train, y_pred_train_lgbm))
wmape_train_lgbm  = np.sum(np.abs(y_train - y_pred_train_lgbm)) / np.sum(np.abs(y_train))
r2_train_lgbm = r2_score(y_train, y_pred_train_lgbm)

print("LightGBM Model Performance on Training Data:")
print(f"MAE: {mae_train_lgbm}")
print(f"RMSE: {rmse_train_lgbm}")
print(f"WMAPE: {wmape_train_lgbm :.4f}")
print(f"R²: {r2_train_lgbm}")


LightGBM Model Performance on Training Data:
MAE: 163.44550040491
RMSE: 244.42664790133065
WMAPE: 0.0362
R²: 0.9945029061994617


In [52]:
y_pred = wind_model.predict(X_valid)

mae_train_lgbm = mean_absolute_error(y_valid, y_pred)
rmse_train_lgbm = np.sqrt(mean_squared_error(y_valid, y_pred))
r2_train_lgbm = r2_score(y_valid, y_pred)
wmape = np.sum(np.abs(y_valid - y_pred)) / np.sum(np.abs(y_valid))
print("LightGBM Model Performance on Training Data:")
print(f"MAE: {mae_train_lgbm}")
print(f"RMSE: {rmse_train_lgbm}")
print(f"WMAPE: {wmape:.4f}")
print(f"R²: {r2_train_lgbm}")


LightGBM Model Performance on Training Data:
MAE: 959.2634580738224
RMSE: 1404.9042318831273
WMAPE: 0.1834
R²: 0.8717049288044991


Overfitting!

## 2. XGBoost

In [53]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [54]:
y_pred_train_xgb = xgb_model.predict(X_train)

mae_train_xgb = mean_absolute_error(y_train, y_pred_train_xgb)
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))
wmape_train_xgb  = np.sum(np.abs(y_train - y_pred_train_xgb)) / np.sum(np.abs(y_train))
r2_train_xgb = r2_score(y_train, y_pred_train_xgb)

print("XGBoost Model Performance on Training Data:")
print(f"MAE: {mae_train_xgb}")
print(f"RMSE: {rmse_train_xgb}")
print(f"WMAPE: {wmape_train_xgb :.4f}")
print(f"R²: {r2_train_xgb}")

XGBoost Model Performance on Training Data:
MAE: 217.64300537109375
RMSE: 297.7403178526549
WMAPE: 0.0482
R²: 0.9918433427810669


In [55]:
y_pred_xgb = xgb_model.predict(X_valid)

mae_valid_xgb = mean_absolute_error(y_valid, y_pred_xgb)
rmse_valid_xgb = np.sqrt(mean_squared_error(y_valid, y_pred_xgb))
wmape_valid_xgb = np.sum(np.abs(y_valid - y_pred_xgb)) / np.sum(np.abs(y_valid))
r2_valid_xgb = r2_score(y_valid, y_pred_xgb)

print("XGBoost Model Performance on Validation Data:")
print(f"MAE: {mae_valid_xgb}")
print(f"RMSE: {rmse_valid_xgb}")
print(f"WMAPE: {wmape_valid_xgb:.4f}")
print(f"R²: {r2_valid_xgb}")

XGBoost Model Performance on Validation Data:
MAE: 961.844482421875
RMSE: 1404.0204770586504
WMAPE: 0.1839
R²: 0.8718662858009338




Both models exhibit signs of overfitting, as their performance on the training data is much better than on the validation data. This could be addressed by techniques like hyperparameter tuning, using more regularization, or increasing the amount of training data if possible.

Overall, while both models perform well, neither is significantly superior to the other on the validation set without further tuning. The choice between them might come down to computational efficiency or specific business requirements.

Cross-validation


In [56]:
tscv = TimeSeriesSplit(n_splits=3)
X_cv = wind_df.drop('FR_wind_actual', axis=1)
y_cv = wind_df['FR_wind_actual']


In [57]:
rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

rmse_scores = cross_val_score(xgb_model, X_cv, y_cv, cv=tscv, scoring=rmse_scorer)
r2_scores = cross_val_score(xgb_model, X_cv, y_cv, cv=tscv, scoring='r2')

print(f"\nCross-Validation Results:")
print(f"RMSE per fold: {-rmse_scores}")
print(f"Average RMSE: {-rmse_scores.mean():.2f}")
print(f"R² per fold: {r2_scores}")
print(f"Average R²: {r2_scores.mean():.4f}")


Cross-Validation Results:
RMSE per fold: [ 823.66911591 1096.87294159 1485.4453036 ]
Average RMSE: 1135.33
R² per fold: [0.91520035 0.89031726 0.87164253]
Average R²: 0.8924
